# Notebook 02 — LLMs as Research Instruments



**Duration:** ~20 minutes



In notebook 01 you set up your tools and confirmed that the university machine is ready. In this notebook you send your first message to the large language model (LLM). Then we look at the model more carefully: what does it already know, where does its knowledge break down, and why does it sometimes give different answers to the same question?



An LLM is a system trained on a very large amount of text. It learns patterns from that text — facts, grammar, reasoning styles — and uses those patterns to generate new text. It does not look anything up. It does not search the internet. Everything it produces comes from patterns it absorbed during training.



This makes it useful, but also unreliable in specific ways. By the end of this notebook you will understand those limitations well enough to work with them.

## Setup

Run this cell first. It loads your base URL and API key from the `.env` file you set up in notebook 01.

In [2]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

import os, json, base64, requests, io
from dotenv import load_dotenv
from openai import OpenAI
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load your base URL and API key from the .env file (set up in notebook 01)
load_dotenv(override=True)
if not os.getenv("LLM_BASE_URL") or not os.getenv("LLM_API_KEY"):
    raise RuntimeError("Could not find LLM_BASE_URL or LLM_API_KEY. Check your .env file (see notebook 01).")

client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# Each Dell Pro Max machine serves one model. This is its name.
# It reads both text and images.
MODEL = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4"

# Turn off the model's "thinking" step. Answers come back several times faster,
# which matters when many people share one machine.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}

print(f"Using model {MODEL}.")
print("Setup complete.")

Using model nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4.
Setup complete.


## Exercise a — Your first message to the model

Let's send a short message to the model and read its reply. This is the simplest possible call: we tell it which model to use, and what to say.

We will then go through the syntax to understand how it works.

Run the cell below.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.choices[0].message.content)


You should see `Paris`, probably in a short sentence. The exact wording may differ from your neighbour's — that is expected. We will explore why later in this notebook.

<details>
<summary><strong>Let's break down what just happened</strong></summary>

| Line | What it does |
|------|--------------|
| `response` | Stores the entire response from the API |
| `client.chat.completions.create(...)` | Sends a message to the LLM on the Dell Pro Max |
| `model=MODEL` | Tells it which model to use (we set this in the setup cell) |
| `messages=[{...}]` | The conversation — a list of messages, each with a `role` and `content` |
| `"role": "user"` | Labels who is speaking — `"user"` means you, the human asking the question |
| `"content": "..."` | The actual text you are sending |
| `response.choices[0].message.content` | Extracts the model's reply from the response |

You will see this pattern in every notebook. The only thing that changes is the message you send.

</details>


### Your turn

Change the question below to anything you like and run the cell. Try asking it something related to your own research area.

This cell has one addition: `extra_body=NO_THINKING`. Some LLMs, including ours, write hidden working notes before they answer. This is called "thinking". It makes answers slower, which matters when many people share one machine. `NO_THINKING` was defined in the setup cell and switches it off. From here on, every call in the workshop includes it.


In [8]:
# Change the question to anything you like.
# Some ideas:
#   "What is thematic analysis in qualitative research?"
#   "Summarise the purpose of the NZ Human Rights Act 1993 in two sentences."
#   "What are the ethical risks of using AI in social science research?"

my_question = "What qualitative research methods could be used in the area of New Zealand environmental protection?"

response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,   # switch off the model's hidden "thinking" step
    messages=[
        {"role": "user", "content": my_question}
    ]
)

print(response.choices[0].message.content)



Below is a practical “tool‑kit” of qualitative research methods that fit well with the kinds of questions that arise in **New Zealand environmental‑protection research** – from policy‐making and stakeholder engagement to understanding Māori‑centred worldviews and the lived experiences of communities that live alongside natural resources.

| Method | Why it fits NZ environmental protection | Typical data sources & products | Key considerations for NZ context |
|--------|----------------------------------------|--------------------------------|-----------------------------------|
| **In‑depth (semi‑structured) interviews** | Capture personal narratives, knowledge systems, and values that shape attitudes toward conservation, resource use, and regulation. | Transcripts; interview guides; coding matrices. | Use both English and te reo Māori versions; consider iwi‑specific cultural protocols (e.g., whakawhanaungatanga, manaakitanga). |
| **Focus groups** | Good for exploring collective perc

## What does the model already know?

The model was trained on a broad range of text — books, websites, legal documents, academic papers. That means it has **implicit knowledge**: things it absorbed during training, without being given a specific source to read.

This is useful. But it is also risky, because you cannot tell which facts it learned accurately and which ones it reconstructed imperfectly.

Let's test this. We will ask the model about two real New Zealand Acts — without giving it any source text to read.

## Exercise b — A well-known Act

The Privacy Act 2020 is widely discussed in legal and policy writing. The model has almost certainly seen many documents about it during training.

Run the cell below. We are not giving the model any data — just asking what it already knows.

In [3]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    messages=[
        {"role": "user", "content": "What is the purpose of New Zealand's Privacy Act 2020?"}
    ]
)

print(response.choices[0].message.content)


The purpose of New Zealand's Privacy Act 2020 is to protect individuals' personal information and privacy rights by setting out rules for how personal information is collected, used, disclosed, and stored. The Act aims to ensure that individuals have control over their personal information and that organizations handle this information responsibly and transparently. The Privacy Act 2020 also promotes fairness, accountability, and transparency in the handling of personal information, and provides individuals with rights and remedies to protect their privacy.


You should see a confident, detailed answer. The model sounds like it knows this Act well.

Now let's try something less well-known.

## Exercise c — An obscure Act

The Impounding Act 1955 is a real New Zealand Act that is still in force. It deals with a very specific (and quite old-fashioned) area of law. It is unlikely to appear in many modern training documents.

We will ask the model two questions about it — without giving it any source text. See how it goes.

In [6]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    messages=[
        {"role": "user", "content": "What is the main purpose of New Zealand's Impoundment Act 1955?"}
    ]
)

print(response.choices[0].message.content)


The Impoundment Act 1955 in New Zealand is designed to regulate the impoundment of animals, particularly livestock, to ensure that they are held securely and safely. This legislation outlines the procedures for impounding animals, the responsibilities of those who impound them, and the rights of the animal owners. The Act aims to prevent livestock from causing damage or posing a risk to public safety by being at large, while also providing a framework for the humane treatment and eventual release or disposal of impounded animals. This includes provisions for the payment of rates or fees for the impoundment and care of the animals, as well as the process for reclaiming the animals by their owners. The Act is part of New Zealand's broader legal framework for animal management and control, which is intended to protect both the community and the welfare of animals.


In [5]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    messages=[
        {"role": "user", "content": "New Zealand's Impoundment Act 1955, what is the maximum fine for not discosing the destruction of wandering livestock onto your property given in `Special remedies for trespass by pigs, goats, or poultry`?"}
    ]
)

print(response.choices[0].message.content)


The maximum fine for not disclosing the destruction of wandering livestock onto your property, as outlined in the "Special remedies for trespass by pigs, goats, or poultry" under New Zealand's Impoundment Act 1955, is $1,000. This fine applies to individuals who fail to report the destruction of livestock that have trespassed onto their property. It is important to note that this information pertains to New Zealand law and may not be applicable in other jurisdictions. For the most accurate and current legal advice, consulting with a legal professional or reviewing the latest version of the legislation is recommended.


Act sources:
- Impoundment ACT: https://www.legislation.govt.nz/act/public/1955/108/en/latest/#DLM293864
- Privacy Act: https://www.legislation.govt.nz/act/public/2020/0031/latest/LMS23223.html

Compare the two answers:

1. Did the model sound equally confident about both Acts?
2. Did it give specific, verifiable details about the Impoundment Act, or did it produce vague, plausible-sounding text?
3. Would you trust either answer without checking the source?

"Hallucinations"
What we've seen is **implicit knowledge** in action. The model does not tell you when it is guessing. It produces fluent text regardless of whether its information is accurate. For well-documented topics it is often right. For obscure ones it may **hallucinate** — generate confident-sounding text that is partially or entirely wrong.

This is why, in notebooks 03–05, we will always give the model **real source data** to work with rather than relying on what it already knows.

## Consistency and temperature

Why do we have different answers to the same question? The model is not a calculator that gives the same output for the same input. It is a text generator that produces different outputs each time, based on **probabilities**.

DIfferent results each run can be an issue for research - which seeks the opposite.

A setting called **temperature** controls how varied LLM answers are:

| Temperature | Behaviour | Use case |
|-------------|-----------|----------|
| `0` | Always picks the most probable word. The output is nearly identical every time. | Research tasks where consistency matters |
| `1` | Picks words with more randomness. The output varies between runs. | Wider range of topics / keyword searches |

## Exercise d — Temperature and repeatability

We will now ask the model the **same question four times**:

- Twice at `temperature=0` (always picks the most probable word)
- Twice at `temperature=1` (more randomness)

We will do this for both Acts, starting with the obscure one. The aim is to find out when the model gives repeatable answers, and whether a repeatable answer is the same thing as a correct one.

To avoid writing eight nearly identical cells, we use two small pieces of Python:

- A **function** called `ask`. It wraps the API call you have already seen, so we can send any question at any temperature in one line.
- A **`for` loop**. It repeats the same block of code for each value in a list. `for temperature in [0, 1]:` runs the block once with `temperature` set to `0`, then once with it set to `1`.

Run the cell below to define the function and the two questions. It does not call the model yet.


In [ ]:
def ask(prompt, temperature):
    """Send one question to the model and return its reply as text."""
    response = client.chat.completions.create(
        model=MODEL,
        extra_body=NO_THINKING,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# The same style of question for each Act: name a specific part of the Act and ask for its contents.
IMPOUNDING_PROMPT = (
    "From New Zealand's Impounding Act 1955, list the special remedies for trespass "
    "by pigs, goats, or poultry. Be concise."
)

PRIVACY_PROMPT = (
    "From New Zealand's Privacy Act 2020, list the information privacy principles. Be concise."
)

print("Function and prompts defined.")


### Part 1 — The obscure Act

First, the Impounding Act 1955. The loop below asks the same question four times: two runs at `temperature=0`, then two runs at `temperature=1`.

As the answers come in, keep an eye on two things:

1. Are the two `temperature=0` answers the same as each other?
2. Do the two `temperature=1` answers agree with each other **on the facts**, or only on the general topic?


In [ ]:
for temperature in [0, 1]:
    for run in [1, 2]:
        print(f"=== Impounding Act 1955 | temperature = {temperature} | run {run} ===")
        print(ask(IMPOUNDING_PROMPT, temperature))
        print()


The two `temperature=0` answers should be identical, or very close. That looks reassuring. But look at the two `temperature=1` answers: they probably list different remedies, different section numbers, or different penalties. When the model is free to vary its answer, the *facts* vary, not just the wording. That is a sign that it is reconstructing plausible-sounding text rather than recalling something it learned well.

You can check any of these answers against the real Act: https://www.legislation.govt.nz/act/public/1955/108/en/latest/whole.html

### Part 2 — The well-known Act

Now the same experiment with the Privacy Act 2020. The question has the same shape: name a specific part of the Act and ask for its contents.


In [ ]:
for temperature in [0, 1]:
    for run in [1, 2]:
        print(f"=== Privacy Act 2020 | temperature = {temperature} | run {run} ===")
        print(ask(PRIVACY_PROMPT, temperature))
        print()


### What did you notice?

Compare the two experiments.

| | Impounding Act 1955 (obscure) | Privacy Act 2020 (well-known) |
|---|---|---|
| `temperature=0`, two runs | Same words each time | Same words each time |
| `temperature=1`, two runs | Different wording **and different facts** | Different wording, but the same principles |
| Are the answers correct? | Check the source — probably not | Check the source — probably yes |

Two lessons come out of this.

**Temperature controls repeatability, not accuracy.** At `temperature=0` the model gives the same answer every time, whether or not that answer is right. For the Impounding Act, you get the same *wrong* answer twice. A repeatable result is not evidence that the model knows what it is talking about.

**Consistency across `temperature=1` runs tells you something about training.** When the model has seen a topic many times in training, the underlying facts are stable and only the phrasing changes between runs. When it has barely seen the topic, each run produces a different plausible-sounding reconstruction. The Privacy Act is discussed in thousands of documents. The Impounding Act is not.

For research this matters in both directions. If you used `temperature=0` to classify 200 legislative sections, you would get the same classification every run. That is good for reproducibility. But if the model does not know the material, you would reproduce the same mistakes 200 times. This is why, in notebooks 03 to 05, we give the model the real source text to work from rather than relying on what it remembers.

You can check the Privacy Act answers here: https://www.legislation.govt.nz/act/public/2020/0031/latest/whole.html


## Exercise e — Inconsistency versus creativity

So far we have treated `temperature=1` as a problem — a source of inconsistency to be avoided. But that is not the whole story. Higher temperature makes the model produce a wider spread of answers. For some research tasks, that spread is exactly what you want.

Imagine you have a passage of text and you want to come up with keywords to code it. A single run at `temperature=0` gives you one consistent answer — good for reproducibility, but you might miss useful alternatives. Running at `temperature=1` gives you a different menu of candidate keywords to consider.

**First, you try.** Read the passage below and pick **10 keywords** that you think capture the substantive concepts. Then we will ask the model to do the same at `temperature=0` and `temperature=1`, and look at the three lists side by side.

In [ ]:
PRIVACY_ACT_PASSAGE = """
What is personal information and the Privacy Act? 

Personal information is “information about an identifiable individual”. It covers both information that is simply about a person (e.g. eye colour) and information that may also identify them (e.g. their name). The information does not need to name the individual, as long as they are identifiable in other ways, like through their home address.

The Privacy Act 2020 provides the rules in New Zealand for protecting personal information and puts responsibilities on agencies and organizations about how they must do that. For example, people have a right to know what information your agency holds about them and a right to ask you to correct it if they think it is wrong.
"""

print(PRIVACY_ACT_PASSAGE)

# Your turn — replace these with 10 keywords you pick from the passage.
my_keywords = [
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
]

In [ ]:
keyword_prompt = (
    "Extract 10 keywords that capture the substantive concepts in this passage. "
    "Return ONLY a JSON array of lowercase strings, e.g. [\"consent\", \"disclosure\"]. "
    "No preamble. No code fences.\n\n"
    f"Passage:\n{PRIVACY_ACT_PASSAGE}"
)

def extract_keywords(temperature):
    response = client.chat.completions.create(
        model=MODEL,
        extra_body=NO_THINKING,
        temperature=temperature,
        messages=[{"role": "user", "content": keyword_prompt}]
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = "\n".join(l for l in raw.splitlines() if not l.startswith("```")).strip()
    return [k.lower().strip() for k in json.loads(raw)]

temp0_keywords = extract_keywords(0)
temp1_keywords = extract_keywords(1)

print("Temperature = 0:", temp0_keywords)
print()
print("Temperature = 1:", temp1_keywords)

In [ ]:
# Side-by-side comparison of the three lists.
rows = max(len(my_keywords), len(temp0_keywords), len(temp1_keywords))

header = f"{'Your keywords':<25} {'Temp = 0':<25} {'Temp = 1':<25}"
print(header)
print("-" * len(header))
for i in range(rows):
    a = my_keywords[i]    if i < len(my_keywords)    else ""
    b = temp0_keywords[i] if i < len(temp0_keywords) else ""
    c = temp1_keywords[i] if i < len(temp1_keywords) else ""
    print(f"{a:<25} {b:<25} {c:<25}")

### When is higher temperature actually useful?

Compare the three lists. Your own picks, `temperature=0`, and `temperature=1` will overlap in places and diverge in others:

- At `temperature=0`, you get one consistent list. Great for reproducibility — run it again and you get the same keywords.
- At `temperature=1`, you get a different spread. Some framings show up here that `temperature=0` would never surface.

This flips the usual advice. For **coding and extraction** — where consistency is everything — stick with `temperature=0`. But for **exploration** — when you want a menu of possible framings, keywords, or interpretations to consider — running at `temperature=1` a few times can surface ideas a single low-temperature run would miss.

## What you accomplished

In this notebook you explored two factors that shape the quality and direction of LLM outputs:

| Factor | What you observed | Exercise |
|--------|-------------------|----------|
| **Training data** | The model was confident on the well-known Privacy Act<br>but unreliable on the obscure Impounding Act.<br>Its accuracy depends on how much relevant,<br>recent data it was trained on. | b & c |
| **Temperature** | Low temperature gives reproducible answers, but reproducible<br>does not mean correct: the obscure Act was wrong the same way<br>twice (exercise d). Higher temperature surfaces a wider spread of<br>candidate framings and keywords (exercise e). | d & e |

The key insight: an LLM is a **research instrument**, not an oracle. Like any instrument, it needs to be calibrated (temperature) and pointed at real data (not just its implicit knowledge). When either of these factors is weak, the output suffers — and the model will not tell you.

This is why, in notebooks 03–05, we default to `temperature=0` for coding and extraction, and reach for higher temperature only when we are still deciding what to look for.

**Next up:** After the break, notebook 03 introduces exploratory analysis. You will build the same request five ways, see how each layer of prompt structure changes the output, and then replicate a published coding method on a passage of real NZ legislation.